# **📊 Pr. 4**

---

**Title:** Message Intelligence System

**Duration:** 6 Hours

---

## **🎯 Objective**

The objective of this project is to design a **classification system** that identifies whether incoming digital messages are **Spam or Legitimate**. Students will combine **probability theory** with **distance-based, margin-based, and probabilistic classifiers**, and analyze how different assumptions impact model performance.

---

## **📑 Problem Statement**

You are working as a **Data Scientist** for a communication security company. The company wants to build a **Message Intelligence System** that can automatically classify user messages as:

- **0 → Legitimate Message**
- **1 → Spam Message**

The dataset contains **message-related features** extracted from text (numerical summaries) and **user behavior signals**. Some features are highly correlated, while others follow probabilistic patterns. Your task is to **build and compare multiple classification models** and explain their performance using **probability concepts**.

---

## **📚 Download and Import Modules**

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB

---

## **🧠 Part A: Conceptual Foundation (Theory)**

Answer briefly (2–4 lines each):

1. What is **Conditional Probability**?
2. Explain **Bayes’ Theorem** and its importance in classification problems.
3. What assumptions does the **Naive Bayes Classifier** make?
4. Explain the working principle of:
   - **K-Nearest Neighbors (KNN)**
   - **Support Vector Machine (SVM)**
5. Compare **distance-based**, **probabilistic**, and **margin-based** classifiers.

Questions & Answers are written in the file named [`theory-concepts.pdf`](./theory-concepts.pdf) in the main directory.

---

## **🧪 Part B: Dataset Understanding & Preparation**

You are provided with a dataset containing:

- Message length  
- Number of special characters  
- Number of URLs  
- Keyword frequency score  
- Sender activity score  
- Time-based features  
- Target variable: **Spam_Label**

**Dataset:** [Access from here](.\data\Message_Intelligence_Dataset_5200.csv)

**Tasks:**

6. Identify input features and target variable.  
7. Perform basic data preprocessing (scaling where required).  
8. Split the dataset into **training and testing sets**.

In [2]:
# Load the dataset
df = pd.read_csv("./data/Message_Intelligence_Dataset_5200.csv")

# 6. Identify input features and target variable
# Exclude non-numeric / identifier columns from model inputs
exclude_cols = ["message_id", "message_text", "timestamp", "spam_label"]
feature_cols = [col for col in df.columns if col not in exclude_cols]

X = df[feature_cols]
y = df["spam_label"]

print("Input features:", feature_cols)
print("Target variable: spam_label")

# 7. Basic preprocessing (scaling)
# Standardization is useful for distance- and margin-based models such as KNN and SVM
X = X.copy()
X = X.apply(pd.to_numeric, errors="coerce")
X = X.fillna(X.mean(numeric_only=True))

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled_df = pd.DataFrame(X_scaled, columns=feature_cols, index=df.index)

# 8. Split into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled_df,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("\nTraining set shape:", X_train.shape, y_train.shape)
print("Testing set shape:", X_test.shape, y_test.shape)
print("\nTraining label distribution:\n", y_train.value_counts(normalize=True))
print("\nTesting label distribution:\n", y_test.value_counts(normalize=True))

Input features: ['message_length', 'word_count', 'num_urls', 'num_digits', 'num_special_chars', 'spam_keyword_score', 'legit_keyword_score', 'sender_activity_score', 'sender_account_age_days', 'messages_sent_last_24h', 'hour_of_day', 'day_of_week']
Target variable: spam_label

Training set shape: (4160, 12) (4160,)
Testing set shape: (1040, 12) (1040,)

Training label distribution:
 spam_label
0    0.812981
1    0.187019
Name: proportion, dtype: float64

Testing label distribution:
 spam_label
0    0.8125
1    0.1875
Name: proportion, dtype: float64


---

## **📊 Part C: Baseline Model – K-Nearest Neighbors**

9. Implement **K-Nearest Neighbors (KNN)** classifier.  
10. Experiment with different values of **K**.  
11. Analyze how distance metrics affect predictions.  
12. Identify cases where KNN misclassifies messages.

In [3]:
# 9. Implement KNN classifiers and 10. Experiment with different values of K
k_values = [1, 3, 5, 7, 9]
metrics = ["euclidean", "manhattan"]
results = []

for metric in metrics:
    for k in k_values:
        knn = KNeighborsClassifier(n_neighbors=k, metric=metric)
        knn.fit(X_train, y_train)
        pred = knn.predict(X_test)

        results.append({
            "metric": metric,
            "k": k,
            "accuracy": round(accuracy_score(y_test, pred), 4),
            "precision": round(precision_score(y_test, pred, zero_division=0), 4),
            "recall": round(recall_score(y_test, pred, zero_division=0), 4),
            "f1": round(f1_score(y_test, pred, zero_division=0), 4),
        })

results_df = pd.DataFrame(results)
print("KNN performance comparison:\n")
print(results_df.sort_values(["metric", "k"]).to_string(index=False))

# Best model based on accuracy
best_rows = results_df.loc[results_df.groupby("metric")["accuracy"].idxmax()]
print("\nBest model by accuracy per metric:")
print(best_rows.to_string())

# 11. Analyze how distance metrics affect predictions
summary = results_df.groupby("metric").agg({
    "accuracy": ["mean", "max"],
    "precision": "mean",
    "recall": "mean",
    "f1": "mean"
})
print("\nDistance-metric summary:\n")
print(summary)

# 12. Identify cases where KNN misclassifies messages
best_metric = best_rows.iloc[0]["metric"]
best_k = int(best_rows.iloc[0]["k"])

best_knn = KNeighborsClassifier(n_neighbors=best_k, metric=best_metric)
best_knn.fit(X_train, y_train)
pred_best = best_knn.predict(X_test)

misclassified_mask = y_test != pred_best
misclassified_index = y_test.index[misclassified_mask]
misclassified_pred = pd.Series(pred_best[misclassified_mask], index=misclassified_index)

print(f"\nMisclassified examples for best KNN ({best_metric}, k={best_k}):")
if len(misclassified_index) > 0:
    sample = df.loc[misclassified_index[:10], ["message_text", "spam_label", "message_length", "word_count", "num_urls", "num_digits", "num_special_chars", "spam_keyword_score", "legit_keyword_score"]]
    sample["predicted_label"] = misclassified_pred.loc[sample.index].values
    print(sample.to_string(index=False))
else:
    print("No misclassifications found.")

# Also show confusion matrix for the best model
cm = confusion_matrix(y_test, pred_best)
print("\nConfusion matrix:")
print(cm)

KNN performance comparison:

   metric  k  accuracy  precision  recall     f1
euclidean  1     1.000        1.0  1.0000 1.0000
euclidean  3     1.000        1.0  1.0000 1.0000
euclidean  5     1.000        1.0  1.0000 1.0000
euclidean  7     1.000        1.0  1.0000 1.0000
euclidean  9     0.999        1.0  0.9949 0.9974
manhattan  1     1.000        1.0  1.0000 1.0000
manhattan  3     1.000        1.0  1.0000 1.0000
manhattan  5     1.000        1.0  1.0000 1.0000
manhattan  7     1.000        1.0  1.0000 1.0000
manhattan  9     1.000        1.0  1.0000 1.0000

Best model by accuracy per metric:
      metric  k  accuracy  precision  recall   f1
0  euclidean  1       1.0        1.0     1.0  1.0
5  manhattan  1       1.0        1.0     1.0  1.0

Distance-metric summary:

          accuracy      precision   recall       f1
              mean  max      mean     mean     mean
metric                                             
euclidean   0.9998  1.0       1.0  0.99898  0.99948
manhattan  

---

## **⚙️ Part D: Support Vector Machine Classifier**

13. Implement **Support Vector Machine (SVM)** classifier with:  
    - Linear kernel  
    - RBF or Polynomial kernel  
14. Analyze margin separation and support vectors.  
15. Compare SVM performance with KNN.

In [4]:
# 13. Implement SVM classifiers with different kernels
svm_models = {
    "linear": SVC(kernel="linear", random_state=42),
    "rbf": SVC(kernel="rbf", random_state=42),
    "poly": SVC(kernel="poly", degree=3, random_state=42)
}

svm_results = []

for name, model in svm_models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    svm_results.append({
        "kernel": name,
        "accuracy": round(accuracy_score(y_test, pred), 4),
        "precision": round(precision_score(y_test, pred, zero_division=0), 4),
        "recall": round(recall_score(y_test, pred, zero_division=0), 4),
        "f1": round(f1_score(y_test, pred, zero_division=0), 4),
        "support_vectors": model.n_support_.sum(),
        "decision_function_shape": model.decision_function_shape,
    })

svm_results_df = pd.DataFrame(svm_results)
print("SVM performance comparison:\n")
print(svm_results_df.to_string(index=False))

# 14. Analyze margin separation and support vectors
best_svm_name = svm_results_df.sort_values("accuracy", ascending=False).iloc[0]["kernel"]
best_svm = svm_models[best_svm_name]

print(f"\nBest SVM kernel: {best_svm_name}")
print("Number of support vectors:", best_svm.n_support_)
print("Total support vectors:", best_svm.n_support_.sum())
print("Decision function shape:", best_svm.decision_function_shape)

# 15. Compare SVM performance with KNN
knn_best = results_df.loc[(results_df["metric"] == best_metric) & (results_df["k"] == best_k)]
knn_row = knn_best.iloc[0]

comparison_df = pd.DataFrame([
    {
        "model": "KNN",
        "metric": best_metric,
        "k": best_k,
        "accuracy": knn_row["accuracy"],
        "precision": knn_row["precision"],
        "recall": knn_row["recall"],
        "f1": knn_row["f1"],
    },
    {
        "model": "SVM",
        "metric": best_svm_name,
        "k": None,
        "accuracy": svm_results_df.loc[svm_results_df["kernel"] == best_svm_name, "accuracy"].iloc[0],
        "precision": svm_results_df.loc[svm_results_df["kernel"] == best_svm_name, "precision"].iloc[0],
        "recall": svm_results_df.loc[svm_results_df["kernel"] == best_svm_name, "recall"].iloc[0],
        "f1": svm_results_df.loc[svm_results_df["kernel"] == best_svm_name, "f1"].iloc[0],
    }
])

print("\nSVM vs KNN comparison:\n")
print(comparison_df.to_string(index=False))


SVM performance comparison:

kernel  accuracy  precision  recall  f1  support_vectors decision_function_shape
linear       1.0        1.0     1.0 1.0                9                     ovr
   rbf       1.0        1.0     1.0 1.0              141                     ovr
  poly       1.0        1.0     1.0 1.0               67                     ovr

Best SVM kernel: linear
Number of support vectors: [5 4]
Total support vectors: 9
Decision function shape: ovr

SVM vs KNN comparison:

model    metric   k  accuracy  precision  recall  f1
  KNN euclidean 1.0       1.0        1.0     1.0 1.0
  SVM    linear NaN       1.0        1.0     1.0 1.0


---

## **🧮 Part E: Naive Bayes Classifier & Probability**

16. Implement **Naive Bayes Classifier**.  
17. Manually compute conditional probabilities for a few sample messages.  
18. Demonstrate how **Bayes’ Theorem** is applied to compute class probabilities.  
19. Compare theoretical probability calculations with model predictions.

In [5]:
# 16. Implement Naive Bayes Classifier
nb = GaussianNB()
nb.fit(X_train, y_train)
pred_nb = nb.predict(X_test)

nb_accuracy = accuracy_score(y_test, pred_nb)
nb_precision = precision_score(y_test, pred_nb, zero_division=0)
nb_recall = recall_score(y_test, pred_nb, zero_division=0)
nb_f1 = f1_score(y_test, pred_nb, zero_division=0)

print("Naive Bayes performance:")
print(f"Accuracy: {nb_accuracy:.4f}")
print(f"Precision: {nb_precision:.4f}")
print(f"Recall: {nb_recall:.4f}")
print(f"F1-score: {nb_f1:.4f}")
print("\nConfusion matrix:")
print(confusion_matrix(y_test, pred_nb))

# 17. Manually compute conditional probabilities for a few sample messages
# Use the first few rows from the test set as sample messages
sample_rows = X_test.iloc[:3].copy()
print("\nSample messages (feature values):")
print(sample_rows.round(3).to_string())

# 18. Demonstrate how Bayes' Theorem is applied to compute class probabilities
# For each sample, compute P(class | features) using GaussianNB's fitted parameters
sample_predictions = []
for idx, row in sample_rows.iterrows():
    x = row.to_numpy().reshape(1, -1)
    class_prior = nb.class_prior_
    means = nb.theta_
    variances = nb.var_

    log_posteriors = []
    for c in range(len(nb.classes_)):
        diff = x[0] - means[c]
        variance = np.maximum(variances[c], 1e-9)
        log_likelihood = -0.5 * (
            len(diff) * np.log(2 * np.pi) + np.sum(np.log(variance)) + np.sum((diff ** 2) / variance)
        )
        log_posteriors.append(np.log(class_prior[c]) + log_likelihood)

    log_posteriors = np.array(log_posteriors)
    log_posteriors -= np.max(log_posteriors)
    probs = np.exp(log_posteriors)
    probs = probs / probs.sum()

    sample_predictions.append({
        "row_index": idx,
        "class_probabilities": dict(zip(nb.classes_, probs)),
        "predicted_class": nb.classes_[np.argmax(probs)]
    })

print("\nManual Bayes-style class probabilities for sample messages:")
for item in sample_predictions:
    print(f"Row {item['row_index']}: {item['class_probabilities']}")
    print(f"Predicted class: {item['predicted_class']}")

# 19. Compare theoretical probability calculations with model predictions
# Compare manual posterior-based predictions with scikit-learn predictions
manual_preds = [item["predicted_class"] for item in sample_predictions]
model_preds = pred_nb[:3]

print("\nComparison of manual Bayes probabilities vs model predictions:")
for i, (manual, model) in enumerate(zip(manual_preds, model_preds)):
    print(f"Sample {i+1}: manual={manual}, model={model}")


Naive Bayes performance:
Accuracy: 1.0000
Precision: 1.0000
Recall: 1.0000
F1-score: 1.0000

Confusion matrix:
[[845   0]
 [  0 195]]

Sample messages (feature values):
      message_length  word_count  num_urls  num_digits  num_special_chars  spam_keyword_score  legit_keyword_score  sender_activity_score  sender_account_age_days  messages_sent_last_24h  hour_of_day  day_of_week
799            0.720       1.580     -0.51      -0.696             -0.447              -0.364                0.328                 -1.544                   -0.077                  -0.419        1.235       -0.504
655           -1.047      -0.284     -0.51      -0.696              1.387              -0.364               -1.469                 -0.796                   -1.496                   2.309       -0.076        1.436
5075          -0.404      -0.284     -0.51      -0.696             -0.447              -0.364                0.328                 -0.162                    0.613                  -0.964      

---

## **📈 Part F: Model Comparison & Evaluation**

20. Evaluate all models using appropriate classification metrics:
    - Accuracy
    - Precision
    - Recall
    - F1 Score
21. Compare:
    - KNN vs SVM vs Naive Bayes
22. Identify which model performs best for:
    - High precision
    - High recall


In [6]:
# 20. Evaluate all models using appropriate classification metrics
# 21. Compare KNN vs SVM vs Naive Bayes
# 22. Identify which model performs best for high precision and high recall

model_summary = pd.DataFrame([
    {
        "Model": "KNN",
        "Accuracy": round(results_df.loc[(results_df["metric"] == best_metric) & (results_df["k"] == best_k), "accuracy"].iloc[0], 4),
        "Precision": round(results_df.loc[(results_df["metric"] == best_metric) & (results_df["k"] == best_k), "precision"].iloc[0], 4),
        "Recall": round(results_df.loc[(results_df["metric"] == best_metric) & (results_df["k"] == best_k), "recall"].iloc[0], 4),
        "F1 Score": round(results_df.loc[(results_df["metric"] == best_metric) & (results_df["k"] == best_k), "f1"].iloc[0], 4),
    },
    {
        "Model": "SVM",
        "Accuracy": round(svm_results_df.loc[svm_results_df["kernel"] == best_svm_name, "accuracy"].iloc[0], 4),
        "Precision": round(svm_results_df.loc[svm_results_df["kernel"] == best_svm_name, "precision"].iloc[0], 4),
        "Recall": round(svm_results_df.loc[svm_results_df["kernel"] == best_svm_name, "recall"].iloc[0], 4),
        "F1 Score": round(svm_results_df.loc[svm_results_df["kernel"] == best_svm_name, "f1"].iloc[0], 4),
    },
    {
        "Model": "Naive Bayes",
        "Accuracy": round(nb_accuracy, 4),
        "Precision": round(nb_precision, 4),
        "Recall": round(nb_recall, 4),
        "F1 Score": round(nb_f1, 4),
    }
])

print("Model comparison summary:\n")
print(model_summary.to_string(index=False))

# Identify best performers for precision and recall
best_precision_model = model_summary.sort_values("Precision", ascending=False).iloc[0]["Model"]
best_recall_model = model_summary.sort_values("Recall", ascending=False).iloc[0]["Model"]

print("\nBest model for high precision:", best_precision_model)
print("Best model for high recall:", best_recall_model)


Model comparison summary:

      Model  Accuracy  Precision  Recall  F1 Score
        KNN       1.0        1.0     1.0       1.0
        SVM       1.0        1.0     1.0       1.0
Naive Bayes       1.0        1.0     1.0       1.0

Best model for high precision: KNN
Best model for high recall: KNN


---

## **🧾 Part G: Final Analysis & Reporting**

23. Prepare a final report summarizing:  
    - Strengths and weaknesses of each classifier  
    - Impact of probability assumptions in Naive Bayes  
    - Trade-offs between interpretability and performance  
    - Business recommendation for real-world deployment  

24. Submit:  
    - Source code / Jupyter Notebook  
    - Evaluation metrics and plots  
    - Final conclusions

The Summary Report is in the file named [`summary-report.pdf`](./summary-report.pdf) in the main directory.

---